In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [3]:
df = pd.read_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/processed/diabetes_cleaned.csv')
print(df.shape)
df.head()

(101766, 48)


,encounter_id,patient_number,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetes_med,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,African American,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


# Feature Engineering & Modeling
## Day 8 — Feature Engineering
Goals:
- Convert age brackets to numeric
- Create age groups
- Create total visits feature
- Create high risk flag
- Create binary target variable
- Drop ID columns and unused features

In [4]:
age_mapping = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}

df['age_numeric'] = df['age'].map(age_mapping)

# Confirm
print(df[['age', 'age_numeric']].head(10))

        age  age_numeric
0    [0-10)            5
1   [10-20)           15
2   [20-30)           25
3   [30-40)           35
4   [40-50)           45
5   [50-60)           55
6   [60-70)           65
7   [70-80)           75
8   [80-90)           85
9  [90-100)           95


In [5]:
def age_group(age):
    if age <= 40:
        return 'Young'
    elif age <= 60:
        return 'Middle'
    elif age <= 80:
        return 'Senior'
    else:
        return 'Elderly'

df['age_group'] = df['age_numeric'].apply(age_group)

# Confirm
print(df['age_group'].value_counts())

age_group
Senior     48551
Middle     26941
Elderly    19990
Young       6284
Name: count, dtype: int64


### Feature: age_numeric and age_group
- Converted age brackets to numeric midpoint values (e.g. `[70-80)` → 75)
- Created simplified age groups:
  - Young (0-40): 6,284 patients
  - Middle (40-60): 26,941 patients
  - Senior (60-80): 48,551 patients
  - Elderly (80+): 19,990 patients
- Senior group dominates confirming EDA findings

In [6]:
df['total_visits'] = (df['number_inpatient'] + 
                      df['number_outpatient'] + 
                      df['number_emergency'])

# Confirm
print(df['total_visits'].describe())

count    101766.000000
mean          1.202759
std           2.291781
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          80.000000
Name: total_visits, dtype: float64


### Feature: total_visits
- Created by summing `number_inpatient + number_outpatient + number_emergency`
- Mean of ~1.2 total visits per patient
- 50% of patients have 0 prior visits (first time or no recorded history)
- Max of 80 visits — some patients are very frequent utilizers
- High utilizers (large total_visits) likely represent high readmission risk

In [7]:
df['high_risk_flag'] = (
    (df['time_in_hospital'] > 7) & 
    (df['num_medications'] > 20)
).astype(int)

# Confirm
print(df['high_risk_flag'].value_counts())
print(f"\nHigh risk patients: {df['high_risk_flag'].sum()} out of {len(df)}")

high_risk_flag
0    93556
1     8210
Name: count, dtype: int64

High risk patients: 8210 out of 101766


### Feature: high_risk_flag
- Binary flag: 1 if `time_in_hospital > 7` AND `num_medications > 20`
- 8,210 patients flagged as high risk (~8% of dataset)
- 93,556 patients not flagged (0)
- Combines two strongest signals from EDA into a single feature
- Will be interesting to see how this correlates with `<30` readmissions
  in the model

In [8]:
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

# Confirm
print(df['readmitted_binary'].value_counts())
print(f"\nReadmission rate: {df['readmitted_binary'].mean():.2%}")

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

Readmission rate: 11.16%


### Feature: readmitted_binary
- Simplified target variable — 1 if readmitted within 30 days, 0 otherwise
- 11,357 positive cases (readmitted within 30 days) — 11.16%
- 90,409 negative cases — 88.84%
- Dataset is significantly imbalanced — model will need class weighting
  or SMOTE to handle this properly on Day 9
- Using binary target simplifies the problem for the baseline model

In [10]:
df.drop(columns=[
    'encounter_id', 
    'patient_number',
    'diag_1',
    'diag_2', 
    'diag_3',
    'medical_specialty'
], inplace=True)

print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (101766, 47)


In [11]:
df.to_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/processed/diabetes_features.csv', index=False)
print("Saved.")

Saved.


## Day 8 Summary
- Created 5 new features:
  - `age_numeric` — age brackets converted to midpoint numeric values
  - `age_group` — simplified buckets (Young, Middle, Senior, Elderly)
  - `total_visits` — sum of all prior visit types
  - `high_risk_flag` — binary flag for long stay + high medication count
  - `readmitted_binary` — simplified binary target (1 = readmitted <30 days)
- Dropped 6 columns: `encounter_id`, `patient_number`, `diag_1/2/3`,
  `medical_specialty`
- Final shape: 101,766 rows x 47 columns
- Readmission rate: 11.16% — dataset is imbalanced
- Saved to `data/processed/diabetes_features.csv`
- Ready for modeling on Day 9